In [ ]:
#@title Run this cell first -- homework checking setup { display-mode: "form" }
#@markdown **Run this cell before anything else**, every time you open the notebook.
#@markdown <br><br>It sets up `grader.check(...)` so you can check your answers as you go.
#@markdown Nothing is installed and nothing is downloaded -- this notebook is
#@markdown self-contained, so it works even if Colab or GitHub is having a bad day.

import base64
import datetime
import doctest
import json
import sys

RECEIPT_PREFIX = "CS1910-RECEIPT:"

_HTML = """<div style="font-family:system-ui,sans-serif;font-size:14px;margin:4px 0;
 padding:10px 12px;border-left:5px solid {bar};background:{bg};border-radius:4px">
 <b>{name}</b> &mdash; {earned}/{possible} {verdict}{detail}</div>"""


class Result:
    """One question's outcome. Displays as HTML in Colab, as text everywhere else.

    The text form is what lands in the notebook's saved output and what the grader
    parses, so keep the first line machine-readable.
    """

    def __init__(self, name, earned, possible, messages):
        self.name, self.earned, self.possible = name, earned, possible
        self.messages = messages

    @property
    def passed(self):
        return self.earned == self.possible

    def __repr__(self):
        head = "{}: {}/{} {}".format(self.name, self.earned, self.possible,
                                     "PASSED" if self.passed else "FAILED")
        return "\n".join([head] + self.messages)

    def _repr_html_(self):
        bar, bg = ("#2e7d32", "#edf7ed") if self.passed else ("#c62828", "#fdecea")
        verdict = "passed" if self.passed else "not passing yet"
        detail = ""
        if self.messages:
            body = "\n".join(self.messages).replace("&", "&amp;").replace("<", "&lt;")
            detail = ('<pre style="margin:8px 0 0;white-space:pre-wrap;font-size:12px;'
                      'color:#444">{}</pre>'.format(body))
        return _HTML.format(bar=bar, bg=bg, name=self.name, earned=self.earned,
                            possible=self.possible, verdict=verdict, detail=detail)


def run_case(code, globs):
    """Run one doctest string. Returns (passed, report_text).

    `get_doctest` copies `globs`, so a case that rebinds a name cannot leak that
    change back into the student's namespace or into the next case.
    """
    parser = doctest.DocTestParser()
    test = parser.get_doctest(code.rstrip() + "\n", dict(globs), "case", None, 0)
    runner = doctest.DocTestRunner(verbose=False,
                                   optionflags=doctest.NORMALIZE_WHITESPACE)
    out = []
    try:
        runner.run(test, out=out.append, clear_globs=False)
    except Exception as exc:                      # a case that raises is a failure
        return False, "{}: {}".format(type(exc).__name__, exc)
    return runner.failures == 0, "".join(out)


def score(name, spec, globs):
    """Score one question against `globs`. Returns a Result."""
    cases = spec["suites"][0]["cases"] if "suites" in spec else spec["cases"]
    per_case = bool(cases) and all("points" in c for c in cases)

    earned = possible = 0
    failures, notes = [], []
    for case in cases:
        ok, report = run_case(case["code"], globs)
        if per_case:
            possible += case["points"]
            earned += case["points"] if ok else 0
        if ok:
            if case.get("success_message"):
                notes.append(case["success_message"])
        else:
            failures.append(case.get("failure_message") or _trim(report))

    if not per_case:
        possible = spec.get("points", len(cases))
        earned = 0 if failures else possible

    return Result(name, earned, possible, failures or notes)


def _trim(report, limit=600):
    """Doctest reports are verbose; keep the part a first-year student can act on."""
    text = report.strip()
    return text if len(text) <= limit else text[:limit] + "\n... (truncated)"


def caller_namespace():
    """Where the student's variables live.

    In Colab and Jupyter that is the interactive namespace, which is right even when
    `check()` is called from inside a function. Outside IPython -- the headless grader,
    or a plain `python` run -- fall back to the calling frame's globals, which is the
    namespace `exec` was handed.
    """
    try:
        shell = get_ipython()           # noqa: F821  (injected by IPython)
        if shell is not None:
            return shell.user_ns
    except NameError:
        pass
    return sys._getframe(2).f_globals


class Grader:
    """Holds the tests for one homework and remembers what has been checked.

    Named `grader` in the student notebook so existing `grader.check("q1")` cells
    keep working untouched.
    """

    def __init__(self, hw, tests):
        self.hw, self.tests, self.log = hw, tests, {}

    def check(self, name, globs=None):
        if name not in self.tests:
            raise KeyError("no test named {!r} in {} (have: {})".format(
                name, self.hw, ", ".join(sorted(self.tests))))
        result = score(name, self.tests[name], globs or caller_namespace())
        self.log[name] = (result.earned, result.possible)
        return result

    def check_all(self, globs):
        """Score every question against one namespace. This is what grading uses."""
        results = [score(n, self.tests[n], globs) for n in sorted(self.tests)]
        self.log = {r.name: (r.earned, r.possible) for r in results}
        return results

    def summary(self):
        return {"hw": self.hw,
                "results": {k: list(v) for k, v in sorted(self.log.items())},
                "total": sum(e for e, _ in self.log.values()),
                "possible": sum(p for _, p in self.log.values())}

    def receipt(self, student_id="", extra=None):
        """Print the one-line machine-readable receipt and return it.

        The grader re-runs the tests itself, so this line is a convenience for the
        student and a fast triage signal -- never the authoritative score.
        """
        body = self.summary()
        body["id"] = str(student_id)
        body["t"] = datetime.datetime.now().isoformat(timespec="seconds")
        if extra:
            body.update(extra)
        blob = base64.b64encode(
            json.dumps(body, sort_keys=True, separators=(",", ":")).encode()
        ).decode()
        line = RECEIPT_PREFIX + blob
        print(line)
        return line


def parse_receipt(text):
    """Pull the last receipt out of arbitrary text. Returns a dict, or None."""
    found = None
    for chunk in text.split(RECEIPT_PREFIX)[1:]:
        token = chunk.split()[0] if chunk.split() else ""
        try:
            found = json.loads(base64.b64decode(token))
        except Exception:
            continue
    return found


def load_ok_tests(directory):
    """Read a directory of `q*.py` test files (OK_FORMAT doctests) into {name: spec}."""
    import pathlib
    tests = {}
    for path in sorted(pathlib.Path(directory).glob("*.py")):
        namespace = {}
        exec(compile(path.read_text(encoding="utf-8"), str(path), "exec"), namespace)
        spec = namespace["test"]
        tests[spec["name"]] = spec
    return tests

_TESTS = {'q1': {'name': 'q1',
        'points': 3,
        'suites': [{'cases': [{'code': ">>> q1_a.strip().lower() == 'syntax'\nTrue",
                               'hidden': False,
                               'locked': False,
                               'points': 1},
                              {'code': ">>> q1_b.strip().lower() == 'runtime'\nTrue",
                               'hidden': False,
                               'locked': False,
                               'points': 1},
                              {'code': ">>> q1_c.strip().lower() == 'semantic'\nTrue",
                               'hidden': False,
                               'locked': False,
                               'points': 1}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q2': {'name': 'q2',
        'points': 2,
        'suites': [{'cases': [{'code': ">>> q2_function.strip() == 'per_person'\nTrue",
                               'hidden': False,
                               'locked': False,
                               'points': 1},
                              {'code': ">>> q2_error.strip() == 'ZeroDivisionError'\nTrue",
                               'hidden': False,
                               'locked': False,
                               'points': 1}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q3': {'name': 'q3',
        'points': 3,
        'suites': [{'cases': [{'code': '>>> share_of_bill(40.0, 15, 4)\n11.5',
                               'hidden': False,
                               'locked': False,
                               'points': 1},
                              {'code': '>>> share_of_bill(85.0, 18, 7)\n14.33',
                               'hidden': False,
                               'locked': False,
                               'points': 1},
                              {'code': '>>> share_of_bill(50.0, 20, 0) == 0\nTrue',
                               'hidden': False,
                               'locked': False,
                               'points': 1}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q4': {'name': 'q4',
        'points': 1,
        'suites': [{'cases': [{'code': '>>> q4_printed == 21.25\nTrue',
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q5': {'name': 'q5',
        'points': 3,
        'suites': [{'cases': [{'code': '>>> len(q5_normal) == 3 and q5_normal[0] > 0 and '
                                       'q5_normal[1] > 0 and q5_normal[2] >= 2\n'
                                       'True',
                               'hidden': False,
                               'locked': False,
                               'points': 1},
                              {'code': '>>> len(q5_edge) == 3 and q5_edge[2] == 1\nTrue',
                               'hidden': False,
                               'locked': False,
                               'points': 1},
                              {'code': '>>> len(q5_error) == 3 and q5_error[2] == 0\nTrue',
                               'hidden': False,
                               'locked': False,
                               'points': 1}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]}}

grader = Grader('10-Problem-Solving-Debugging', _TESTS)
check = grader.check

print("Ready. 10-Problem-Solving-Debugging loaded: 5 question(s), 12 point(s).")
print("Check a question any time with, for example:  grader.check(\"q1\")")

## Before you begin

### Submission
Before you start working on this Homework:

1. Press **File->Save a copy in Drive** from the menu. This will save your own copy of this notebook

2. Follow the steps below and ensure you run the cells in order from the top. (You'll have to ensure you run any cells above this one too)

3. You should get into the habit of saving your file occassionally as you work

4. Certain cells will check your work against the expected answer, when you run these cells you'll know if your work is correct.

5. After all the changes are done and progress is saved **File->Download->Download.ipynb** and then upload the .ipynb file to moodle

# Homework: Problem Solving and Debugging — Split the Bill

Five of you order in. One card pays, and then everyone argues about the e-transfer for
the rest of the night. So somebody wrote an app: put in the order total, the tip and how
many people are eating, and it tells each of you what to send.

It does not work.

This homework is not about writing a program from scratch. It is about the other half of
the job — the half nobody warns you about — **working out what is wrong with a program
that already exists, and proving you fixed it**.

You will use all four stages of the problem-solving model:

> **Understand** → **Design** → **Implement** → **Evaluate**

and you will spend most of your time in *Understand* and *Evaluate*, which is exactly how
real debugging goes.

## Question 1 — Name the three kinds of error

Every bug you will ever meet is one of three kinds:

- **syntax** — Python cannot even read the code. Nothing runs, not one line.
- **runtime** — the code reads fine and starts running, then blows up partway with a
  traceback.
- **semantic** — the code runs all the way to the end without complaining, and the answer
  is wrong. These are the dangerous ones.

Classify each of the three below. Put `'syntax'`, `'runtime'` or `'semantic'` into
`q1_a`, `q1_b` and `q1_c`.

**A**
```python
def per_person(total, people)
    return total / people
```

**B**
```python
total = "42.50"
people = 3
print(total / people)
```

**C**
```python
def add_tip(total, percent):
    return total + percent

print(add_tip(40, 15))    # prints 55; the bill with a 15% tip is 46
```

Do not guess — paste each one into a scratch cell and watch what Python does. The
difference between "it would not start" and "it started and then stopped" is the whole
distinction.

In [ ]:
q1_a = ...
q1_b = ...
q1_c = ...

In [ ]:
grader.check("q1")

## Question 2 — Read the stack trace

Somebody ran the app with nobody signed up for the order and sent you this:

```
Traceback (most recent call last):
  File "split.py", line 21, in <module>
    print(summary(order_total, headcount))
          ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "split.py", line 15, in summary
    each = per_person(total, people)
           ~~~~~~~~~~^^^^^^^^^^^^^^^
  File "split.py", line 9, in per_person
    return total / people
           ~~~~~~^~~~~~~~
ZeroDivisionError: division by zero
```

A traceback reads **bottom to top**. The bottom line is what went wrong. The frames above
it are the calls that got Python there, most recent last — so the call closest to the
bottom is where the program actually was when it failed.

- Put the name of the function where the error was **raised** in `q2_function`
  (just the name, no brackets).
- Put the exact name of the error in `q2_error` — spelled and capitalised exactly as
  Python spells it.

Then look one frame up. `summary` passed `people` straight through to `per_person`. Notice
that the *cause* of the bug and the *place it exploded* are not the same line. They very
often are not.

In [ ]:
q2_function = ...
q2_error = ...

In [ ]:
grader.check("q2")

## Question 3 — Fix the function

Here is the function at the heart of the app. It is supposed to return **what one person
owes**, tip included, rounded to 2 decimal places.

```python
def share_of_bill(total, tip_percent, people):
    with_tip = total + tip_percent
    return round(with_tip / people, 2)
```

It has **two** bugs — one of each of the interesting kinds:

- a **semantic** bug: `tip_percent` is a *percentage*, not an amount in dollars. A $40
  order with a 15% tip is $46, not $55.
- a **runtime** bug: the `ZeroDivisionError` from Question 2. When `people` is 0, the
  function should return `0` rather than crash.

Rewrite it correctly in the cell below. It must satisfy:

```
share_of_bill(40.0, 15, 4)   ->  11.5     $46 split four ways
share_of_bill(85.0, 18, 7)   ->  14.33    $100.30 split seven ways, rounded
share_of_bill(50.0, 20, 0)   ->  0        nobody to split it between
```

Each of those three is worth 1 point, so fixing one bug and not the other still earns
marks.

**Design before you type.** Write the three steps out in plain language first — check for
nobody, add the tip, divide and round — and then write one line of Python per step.

In [ ]:
def share_of_bill(total, tip_percent, people):
    ...

# try it out
print(share_of_bill(40.0, 15, 4))

In [ ]:
grader.check("q3")

## Question 4 — Debug by inspection

Here is the part of the app that adds up what everyone ordered:

```python
items = [12.50, 8.00, 21.25]
total = 0

for item in items:
    total = item

print(total)
```

The author expected `41.75`. **Read the loop carefully and work out what it actually
prints.** Put that number in `q4_printed`.

Then run it and confirm you were right. Reading code and predicting its behaviour — before
running it — is the single fastest debugging skill there is, and this is a one-character
bug that a `print()` inside the loop would have shown you immediately.

In [ ]:
q4_printed = ...

In [ ]:
grader.check("q4")

## Question 5 — Design the test cases

A fix you cannot demonstrate is not a fix. Before shipping `share_of_bill` you should have
at least three tests:

- a **normal** case — an ordinary, boring, everyday call
- an **edge** case — a legal input right at the boundary of what makes sense
- an **error** case — the input that used to break it

Give one of each, as a **tuple** of the three arguments in order:
`(total, tip_percent, people)`.

The rules each one must satisfy:

| Variable | Must be |
|---|---|
| `q5_normal` | `total` above 0, `tip_percent` above 0, and **2 or more** people |
| `q5_edge` | exactly **1** person — the whole bill lands on one card |
| `q5_error` | exactly **0** people — the case that used to raise `ZeroDivisionError` |

There are many right answers here. Pick numbers you could defend to somebody, then run
your `share_of_bill` on all three and check that each answer is what you expect.

In [ ]:
q5_normal = ...
q5_edge = ...
q5_error = ...

# Uncomment these three lines once you have filled in the tuples above,
# then check that each answer is the one you expected.
# print(share_of_bill(q5_normal[0], q5_normal[1], q5_normal[2]))
# print(share_of_bill(q5_edge[0], q5_edge[1], q5_edge[2]))
# print(share_of_bill(q5_error[0], q5_error[1], q5_error[2]))

In [ ]:
grader.check("q5")

## Beyond the homework

Not graded, but this is where debugging gets real:

- Put `breakpoint()` on the first line of `share_of_bill` and run it. Python drops you
  into `pdb`: type `people` to see the value, `c` to carry on, `q` to quit.
- Open Colab's **variable inspector** (the `{x}` icon in the left sidebar) and watch
  `total` change as the Question 4 loop runs.
- One person paid $46 and three people owe $11.50 each — that only adds to $34.50 from the
  other three. Should `share_of_bill` know the payer is one of the people? Decide what the
  right behaviour is, write the test case for it *first*, and then make it pass.

# Submission

Save a copy in Drive to ensure you have your own copy of this notebook and to ensure that you've saved your latest work.

- Ensure you run all the cells in order from the top to the bottom
- Run the final cell to see your total and produce your receipt
- **File->Download->Download .ipynb** and upload the `.ipynb` file to Moodle

Your mark comes from your instructor re-running this notebook, not from the receipt
line -- so make sure the notebook itself is saved with your work in it.

In [ ]:
#@title Finished? Run this to see your total { display-mode: "form" }
#@markdown Re-checks every question and prints your score, plus a receipt line.
#@markdown <br><br>Your grade comes from your instructor re-running this notebook,
#@markdown so the receipt is for your records -- make sure the notebook itself is saved.

student_id = ""  #@param {type:"string"}

for _r in grader.check_all(globals()):
    print(_r)
_s = grader.summary()
print("\nTOTAL: {}/{}".format(_s["total"], _s["possible"]))
grader.receipt(student_id)
